# A07 + A08 Confirmatory 7-Fold Training — Unattended Kaggle Run

This notebook is adapted directly from the supplied successful A03+A04 Kaggle workflow.

**Run All is safe for unattended training:** it downloads/extracts the same frozen project, audits hashes/environment, preprocesses A07+A08, restores the frozen model definitions, then trains the fixed 7-fold queue with fold-local GLASS and all four neural variants for seeds 1,2,3. Recovery ZIPs are produced by the original restart-safe runner.

**Outer-fold evaluation is intentionally NOT included.** No outer EEG/labels are indexed during this overnight training notebook. After training completes, run the locked outer pre-flight/evaluator separately.


In [ ]:
# ============================================================
# KAGGLE STEP 1 — DOWNLOAD + EXTRACT FROZEN PROJECT
# ============================================================

from pathlib import Path
import subprocess
import sys
import shutil
import hashlib
import os

FILE_ID = "1TM4XHRh-_xd6Eb5hJmlwVk8Xvx5T2Ibs"

ZIP_PATH = Path(
    "/kaggle/working/"
    "EEG_GANet_Reproduction_Kaggle_Minimal.zip"
)

PROJECT_ROOT = Path(
    "/kaggle/working/"
    "EEG_GANet_Reproduction"
)


# ------------------------------------------------------------
# Install gdown
# ------------------------------------------------------------

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "gdown",
    ]
)

import gdown


# ------------------------------------------------------------
# Download
# ------------------------------------------------------------

if not ZIP_PATH.exists():

    print("Downloading frozen project package...")

    gdown.download(
        id=FILE_ID,
        output=str(ZIP_PATH),
        quiet=False
    )

else:

    print("ZIP already present.")


assert ZIP_PATH.is_file(), ZIP_PATH

print(
    "\nDownloaded ZIP:",
    f"{ZIP_PATH.stat().st_size / 1024**2:.2f} MB"
)


# ------------------------------------------------------------
# Extract
# ------------------------------------------------------------

if PROJECT_ROOT.exists():

    shutil.rmtree(
        PROJECT_ROOT
    )

PROJECT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("\nExtracting...")

shutil.unpack_archive(
    str(ZIP_PATH),
    str(PROJECT_ROOT)
)

print("Extraction: PASS")


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

CV_ROOT = (
    PROJECT_ROOT
    / "GLASS_GANet"
    / "results"
    / "paper_confirmatory_cv_v1"
)

print("\nPROJECT_ROOT:")
print(PROJECT_ROOT)

print("\nCV_ROOT:")
print(CV_ROOT)


In [ ]:
# ============================================================
# KAGGLE STEP 2 — FROZEN SOURCE + ENVIRONMENT AUDIT
# ============================================================

import hashlib
import tensorflow as tf
from pathlib import Path


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


DBNET = (
    PROJECT_ROOT
    / "code"
    / "EEG-GANet"
    / "github_model.py"
)

GLASS = (
    PROJECT_ROOT
    / "GLASS_GANet"
    / "source"
    / "GLASS_official"
    / "code"
    / "glass.py"
)

HELPER = (
    PROJECT_ROOT
    / "GLASS_GANet"
    / "source"
    / "GLASS_official"
    / "code"
    / "helper_funcs.py"
)

GLASS_TF220 = (
    PROJECT_ROOT
    / "GLASS_GANet"
    / "integration"
    / "glass_tf220.py"
)


assert DBNET.is_file()
assert GLASS.is_file()
assert HELPER.is_file()
assert GLASS_TF220.is_file()


assert sha256_file(DBNET) == (
    "f3e570d12de34080ea13003a37bf9a2493b93f1a2376631e8224a40942231cd5"
)

assert sha256_file(GLASS) == (
    "00de59d0ba107a7a5770565bdfb5bf7dfbb372f92bed2c4aecd1bb21deda652d"
)

assert sha256_file(HELPER) == (
    "f9cf4a5134acbbda7fcb55e6503096f1698b0f9e63e16a2a4f9252dd51eb54d9"
)


print("=" * 80)
print("KAGGLE FROZEN PROJECT AUDIT")
print("=" * 80)

print("DBNet source hash: PASS")
print("GLASS source hash: PASS")
print("GLASS helper hash: PASS")

print("\nTensorFlow:", tf.__version__)

print(
    "GPU:",
    tf.config.list_physical_devices("GPU")
)


for subject in [
    "A07",
    "A08",
]:

    path = (
        PROJECT_ROOT
        / "dataset"
        / "P300_ALS"
        / f"{subject}.mat"
    )

    assert path.is_file(), path

    print(
        subject,
        "raw dataset: PASS"
    )


print("\n" + "=" * 80)
print("KAGGLE PROJECT AUDIT: PASS")
print("=" * 80)


In [ ]:
# ============================================================
# KAGGLE STEP 3 — RESTART-SAFE SESSION + FROZEN CV SETUP
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import time
import json
import os

# ------------------------------------------------------------
# Session timing
# ------------------------------------------------------------

SESSION_START = time.monotonic()

# Stop well before Kaggle's 12 h hard session limit
SAFE_LIMIT_HOURS = 10.5


def elapsed_hours():
    return (
        time.monotonic()
        - SESSION_START
    ) / 3600.0


def time_remaining_hours():
    return (
        SAFE_LIMIT_HOURS
        - elapsed_hours()
    )


# ------------------------------------------------------------
# Project/result roots
# ------------------------------------------------------------

CV_ROOT = (
    PROJECT_ROOT
    / "GLASS_GANet"
    / "results"
    / "paper_confirmatory_cv_v1"
)

SUBJECT_RUN_ROOT = (
    CV_ROOT
    / "subject_runs"
)

SUBJECT_RUN_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

RECOVERY_ROOT = Path(
    "/kaggle/working/recovery"
)

RECOVERY_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

PROGRESS_FILE = (
    RECOVERY_ROOT
    / "progress_manifest.json"
)


# ------------------------------------------------------------
# Subjects for this Kaggle session
#
# Priority:
#   finish A07 first
#   then continue A08 if time remains
# ------------------------------------------------------------

SUBJECT_QUEUE = [
    "A07",
    "A08",
]


# ------------------------------------------------------------
# Frozen 7-fold CV definitions
# ------------------------------------------------------------

FOLDS = {

    1: {
        "train": [
            1,3,4,6,7,9,10,11,13,14,
            15,16,17,19,20,21,24,25,27,28,
            29,30,31,32,34
        ],
        "val": [
            2,5,12,33,35
        ],
        "outer": [
            8,18,22,23,26
        ],
    },

    2: {
        "train": [
            1,3,4,6,7,8,9,11,13,14,
            15,16,17,18,20,21,22,23,24,26,
            28,29,30,31,34
        ],
        "val": [
            10,19,25,27,32
        ],
        "outer": [
            2,5,12,33,35
        ],
    },

    3: {
        "train": [
            1,2,3,5,6,8,11,12,13,14,
            15,16,18,20,21,22,23,24,26,28,
            29,30,33,34,35
        ],
        "val": [
            4,7,9,17,31
        ],
        "outer": [
            10,19,25,27,32
        ],
    },

    4: {
        "train": [
            1,2,3,5,8,10,11,12,13,14,
            18,19,20,21,22,23,25,26,27,28,
            30,32,33,34,35
        ],
        "val": [
            6,15,16,24,29
        ],
        "outer": [
            4,7,9,17,31
        ],
    },

    5: {
        "train": [
            1,2,3,4,5,7,8,9,10,12,
            13,14,17,18,19,22,23,25,26,27,
            30,31,32,33,35
        ],
        "val": [
            11,20,21,28,34
        ],
        "outer": [
            6,15,16,24,29
        ],
    },

    6: {
        "train": [
            2,4,5,6,7,8,9,10,12,15,
            16,17,18,19,22,23,24,25,26,27,
            29,31,32,33,35
        ],
        "val": [
            1,3,13,14,30
        ],
        "outer": [
            11,20,21,28,34
        ],
    },

    7: {
        "train": [
            2,4,5,6,7,9,10,11,12,15,
            16,17,19,20,21,24,25,27,28,29,
            31,32,33,34,35
        ],
        "val": [
            8,18,22,23,26
        ],
        "outer": [
            1,3,13,14,30
        ],
    },
}


# ------------------------------------------------------------
# Validate frozen splits
# ------------------------------------------------------------

for fold, split in FOLDS.items():

    train = set(split["train"])
    val = set(split["val"])
    outer = set(split["outer"])

    assert len(train) == 25
    assert len(val) == 5
    assert len(outer) == 5

    assert not (train & val)
    assert not (train & outer)
    assert not (val & outer)

    assert (
        train | val | outer
        ==
        set(range(1, 36))
    )


# ------------------------------------------------------------
# Atomic progress writer
# ------------------------------------------------------------

def save_progress(
    subject=None,
    fold=None,
    stage=None,
    seed=None,
    model=None,
    status="running",
):

    payload = {
        "subject": subject,
        "fold": fold,
        "stage": stage,
        "seed": seed,
        "model": model,
        "status": status,

        "elapsed_hours":
            elapsed_hours(),

        "safe_limit_hours":
            SAFE_LIMIT_HOURS,

        "updated_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    tmp = (
        RECOVERY_ROOT
        / "progress_manifest.tmp"
    )

    tmp.write_text(
        json.dumps(
            payload,
            indent=2
        ),
        encoding="utf-8"
    )

    tmp.replace(
        PROGRESS_FILE
    )


# ------------------------------------------------------------
# Completion helpers
# ------------------------------------------------------------

def neural_model_complete(
    subject,
    fold,
    seed,
    model_name
):

    run_dir = (
        SUBJECT_RUN_ROOT
        / subject
        / f"outer_fold_{fold}"
        / f"seed_{seed}"
        / model_name
    )

    return (
        (run_dir / "best.weights.h5").is_file()
        and
        (
            run_dir
            / "training_summary.json"
        ).is_file()
    )


def glass_stage1_complete(
    subject,
    fold
):

    path = (
        SUBJECT_RUN_ROOT
        / subject
        / f"outer_fold_{fold}"
        / "glass"
        / "stage1_no_shrinkage.npz"
    )

    return path.is_file()


def glass_stage2_complete(
    subject,
    fold
):

    path = (
        SUBJECT_RUN_ROOT
        / subject
        / f"outer_fold_{fold}"
        / "glass"
        / "stage2_shrinkage.npz"
    )

    return path.is_file()


# ------------------------------------------------------------
# Initial progress record
# ------------------------------------------------------------

save_progress(
    stage="session_initialized",
    status="ready"
)


print("=" * 80)
print("KAGGLE RESTART-SAFE TRAINING SESSION")
print("=" * 80)

print(
    "Subject queue:",
    SUBJECT_QUEUE
)

print(
    "Safe runtime budget:",
    SAFE_LIMIT_HOURS,
    "hours"
)

print(
    "Frozen folds:",
    len(FOLDS)
)

print(
    "Recovery directory:",
    RECOVERY_ROOT
)

print(
    "Progress manifest:",
    PROGRESS_FILE
)

print("\nOuter-test EEG accessed: NO")
print("Outer performance evaluated: NO")

print("\nKAGGLE STEP 3: PASS")


In [ ]:
# ============================================================
# KAGGLE STEP 4 — PREPROCESS A07 + A08
# FROZEN P300-ALS / GLASS PREPROCESSING
#
# Restart-safe:
#   - verified existing prepared files are reused
#   - completed subject is saved immediately
#
# NO model training
# NO outer-fold evaluation
# ============================================================

from pathlib import Path
import gc
import numpy as np
from scipy.io import loadmat
from scipy import signal


# ============================================================
# 1. PATHS
# ============================================================

DATASET_DIR = (
    PROJECT_ROOT
    / "dataset"
    / "P300_ALS"
)

PREPARED_DIR = (
    PROJECT_ROOT
    / "GLASS_GANet"
    / "prepared_data"
)

PREPARED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert DATASET_DIR.is_dir()


# ============================================================
# 2. FROZEN CONSTANTS
# ============================================================

FS = 256

FILTER_ORDER = 4
FILTER_LOW = 0.1
FILTER_HIGH = 24.0

PRE_SAMPLES = 77
POST_SAMPLES = 256

GLASS_CROP_START = 39
OUTPUT_SAMPLES = 128

N_TRIALS = 35
N_SEQUENCES = 10
N_HALVES = 2
N_CHOICES = 6

EVENTS_PER_SEQUENCE = 12
EVENTS_PER_TRIAL = 120

CHANNEL_NAMES = np.array([
    "Fz",
    "Cz",
    "Pz",
    "Oz",
    "P3",
    "P4",
    "PO7",
    "PO8",
])


FILTER_SOS = signal.butter(
    FILTER_ORDER,
    [FILTER_LOW, FILTER_HIGH],
    btype="bandpass",
    fs=FS,
    output="sos",
)


# ============================================================
# 3. EXACT EVENT GROUPING
# ============================================================

def extract_grouped_events(
    raw_labels,
    raw_stimulus_codes,
    trial_markers,
):

    raw_labels = np.asarray(
        raw_labels
    ).reshape(-1)

    raw_stimulus_codes = np.asarray(
        raw_stimulus_codes
    ).reshape(-1)

    trial_markers = np.asarray(
        trial_markers,
        dtype=np.int64
    ).reshape(-1)

    assert trial_markers.shape == (35,)


    # Positive stimulus begins when its code appears/changes.
    stimulus_changed = np.concatenate([
        np.array([True]),
        (
            raw_stimulus_codes[1:]
            !=
            raw_stimulus_codes[:-1]
        ),
    ])

    all_event_indices = np.flatnonzero(
        stimulus_changed
        &
        (raw_stimulus_codes > 0)
    )


    event_index_grouped = np.empty(
        (35, 10, 2, 6),
        dtype=np.int64
    )

    code_grouped = np.empty(
        (35, 10, 2, 6),
        dtype=np.int16
    )

    y_grouped = np.empty(
        (35, 10, 2, 6),
        dtype=np.int8
    )

    errors = []


    for trial_index in range(35):

        # MATLAB trial marker is one-based.
        trial_start = max(
            0,
            int(
                trial_markers[
                    trial_index
                ]
            ) - 1
        )

        if trial_index + 1 < 35:

            trial_end = (
                int(
                    trial_markers[
                        trial_index + 1
                    ]
                ) - 1
            )

        else:

            trial_end = len(
                raw_stimulus_codes
            )


        trial_events = all_event_indices[
            (all_event_indices >= trial_start)
            &
            (all_event_indices < trial_end)
        ]


        if len(trial_events) != 120:

            errors.append(
                f"Trial {trial_index + 1}: "
                f"{len(trial_events)} events, "
                "expected 120."
            )

            continue


        trial_codes = (
            raw_stimulus_codes[
                trial_events
            ]
            .astype(np.int16)
        )

        trial_labels = (
            raw_labels[
                trial_events
            ]
        )


        for sequence_index in range(10):

            start = (
                sequence_index
                *
                EVENTS_PER_SEQUENCE
            )

            stop = (
                start
                +
                EVENTS_PER_SEQUENCE
            )

            seq_indices = (
                trial_events[start:stop]
            )

            seq_codes = (
                trial_codes[start:stop]
            )

            seq_labels = (
                trial_labels[start:stop]
            )


            if not np.array_equal(
                np.sort(seq_codes),
                np.arange(1, 13)
            ):

                errors.append(
                    f"Trial {trial_index + 1}, "
                    f"sequence {sequence_index + 1}: "
                    "invalid stimulus-code set."
                )

                continue


            # Reorder into:
            #   half 0 = Row1..Row6
            #   half 1 = Col1..Col6
            for code in range(1, 13):

                positions = np.flatnonzero(
                    seq_codes == code
                )

                if len(positions) != 1:

                    errors.append(
                        f"Trial {trial_index + 1}, "
                        f"sequence {sequence_index + 1}, "
                        f"code {code}: "
                        "not exactly one occurrence."
                    )

                    continue


                source_position = int(
                    positions[0]
                )

                if code <= 6:

                    half = 0
                    choice = code - 1

                else:

                    half = 1
                    choice = code - 7


                event_index_grouped[
                    trial_index,
                    sequence_index,
                    half,
                    choice
                ] = seq_indices[
                    source_position
                ]

                code_grouped[
                    trial_index,
                    sequence_index,
                    half,
                    choice
                ] = code


                # Original MATLAB:
                #   2 = target
                #   1 = non-target
                #
                # Frozen grouped label:
                #   1 = target
                #   0 = non-target
                y_grouped[
                    trial_index,
                    sequence_index,
                    half,
                    choice
                ] = int(
                    seq_labels[
                        source_position
                    ] == 2
                )


            row_targets = np.sum(
                (seq_codes >= 1)
                &
                (seq_codes <= 6)
                &
                (seq_labels == 2)
            )

            column_targets = np.sum(
                (seq_codes >= 7)
                &
                (seq_codes <= 12)
                &
                (seq_labels == 2)
            )

            if row_targets != 1:

                errors.append(
                    f"Trial {trial_index + 1}, "
                    f"sequence {sequence_index + 1}: "
                    f"{row_targets} row targets."
                )

            if column_targets != 1:

                errors.append(
                    f"Trial {trial_index + 1}, "
                    f"sequence {sequence_index + 1}: "
                    f"{column_targets} column targets."
                )


    if errors:

        raise RuntimeError(
            "Event validation failed:\n"
            +
            "\n".join(
                errors[:30]
            )
        )


    assert event_index_grouped.shape == (
        35, 10, 2, 6
    )

    assert y_grouped.shape == (
        35, 10, 2, 6
    )

    # One target per six-choice half.
    assert np.all(
        y_grouped.sum(axis=-1) == 1
    )

    # Two targets per complete sequence:
    # one row + one column.
    assert np.all(
        y_grouped.sum(
            axis=(2, 3)
        ) == 2
    )

    # 20 targets per character.
    assert np.all(
        y_grouped.sum(
            axis=(1, 2, 3)
        ) == 20
    )

    assert np.all(
        code_grouped[:, :, 0, :]
        ==
        np.arange(1, 7)
    )

    assert np.all(
        code_grouped[:, :, 1, :]
        ==
        np.arange(7, 13)
    )


    return (
        event_index_grouped,
        code_grouped,
        y_grouped,
    )


# ============================================================
# 4. COMPLETE FROZEN PREPROCESSOR
# ============================================================

def preprocess_subject(subject):

    mat_path = (
        DATASET_DIR
        / f"{subject}.mat"
    )

    assert mat_path.is_file(), mat_path


    mat = loadmat(
        mat_path,
        simplify_cells=True
    )

    assert "data" in mat

    data = mat["data"]


    raw_eeg = np.asarray(
        data["X"],
        dtype=np.float64
    )

    raw_labels = np.asarray(
        data["y"]
    ).reshape(-1)

    raw_stimulus = np.asarray(
        data["y_stim"]
    ).reshape(-1)

    trial_markers = np.asarray(
        data["trial"],
        dtype=np.int64
    ).reshape(-1)

    channels = np.array([
        str(x)
        for x in np.asarray(
            data["channels"]
        ).reshape(-1)
    ])


    # --------------------------------------------------------
    # Raw-data audit
    # --------------------------------------------------------

    assert raw_eeg.ndim == 2
    assert raw_eeg.shape[1] == 8

    assert (
        len(raw_eeg)
        ==
        len(raw_labels)
        ==
        len(raw_stimulus)
    )

    assert trial_markers.shape == (
        35,
    )

    np.testing.assert_array_equal(
        channels,
        CHANNEL_NAMES
    )

    assert np.isfinite(
        raw_eeg
    ).all()


    # --------------------------------------------------------
    # Event extraction/grouping
    # --------------------------------------------------------

    (
        event_index_grouped,
        code_grouped,
        y_grouped,
    ) = extract_grouped_events(
        raw_labels,
        raw_stimulus,
        trial_markers,
    )


    # --------------------------------------------------------
    # Continuous 0.1–24 Hz filter
    # --------------------------------------------------------

    filtered = signal.sosfiltfilt(
        FILTER_SOS,
        raw_eeg,
        axis=0
    )

    assert filtered.shape == (
        raw_eeg.shape
    )

    assert np.isfinite(
        filtered
    ).all()


    # --------------------------------------------------------
    # Epoch extraction
    #
    # offsets = -77 ... +256 inclusive
    # total = 334 samples
    # --------------------------------------------------------

    flat_events = (
        event_index_grouped
        .reshape(-1)
    )

    assert flat_events.shape == (
        4200,
    )

    assert (
        flat_events.min()
        -
        PRE_SAMPLES
    ) >= 0

    assert (
        flat_events.max()
        +
        POST_SAMPLES
    ) < len(filtered)


    offsets = np.arange(
        -PRE_SAMPLES,
        POST_SAMPLES + 1,
        dtype=np.int64
    )

    assert offsets.shape == (
        334,
    )


    epoch_indices = (
        flat_events[:, None]
        +
        offsets[None, :]
    )

    epochs = filtered[
        epoch_indices
    ]

    assert epochs.shape == (
        4200, 334, 8
    )


    # --------------------------------------------------------
    # Baseline correction: -77 through 0 inclusive
    # 78 samples
    # --------------------------------------------------------

    baseline = np.mean(
        epochs[:, :78, :],
        axis=1,
        keepdims=True
    )

    epochs = (
        epochs
        -
        baseline
    )

    assert np.isfinite(
        epochs
    ).all()


    # --------------------------------------------------------
    # 256 Hz -> 128 Hz
    # --------------------------------------------------------

    resampled = signal.resample_poly(
        epochs,
        up=1,
        down=2,
        axis=1
    )

    del epochs
    del baseline

    assert resampled.shape == (
        4200, 167, 8
    )


    # --------------------------------------------------------
    # Frozen GLASS crop: [39:167]
    # --------------------------------------------------------

    processed = (
        resampled[
            :,
            39:167,
            :
        ]
    )

    assert processed.shape == (
        4200, 128, 8
    )


    # epoch × channel × time
    processed = np.transpose(
        processed,
        (0, 2, 1)
    ).astype(np.float32)

    assert processed.shape == (
        4200, 8, 128
    )

    assert np.isfinite(
        processed
    ).all()


    X_grouped = processed.reshape(
        35,
        10,
        2,
        6,
        8,
        128
    )


    return {
        "X_glass_processed_grouped":
            X_grouped.astype(
                np.float32
            ),

        "y_grouped":
            y_grouped.astype(
                np.int8
            ),

        "code_grouped":
            code_grouped.astype(
                np.int16
            ),

        "event_index_grouped":
            event_index_grouped.astype(
                np.int64
            ),

        "trial_markers":
            trial_markers.astype(
                np.int64
            ),

        "channel_names":
            channels,

        "sampling_frequency":
            np.asarray(
                128,
                dtype=np.int32
            ),

        "preprocessing":
            np.asarray(
                "bandpass_0.1_24Hz_order4_"
                "baseline_minus77_to_0_"
                "resample256_to128_"
                "crop39_length128"
            ),
    }


# ============================================================
# 5. RESTART-SAFE SUBJECT PROCESSING
# ============================================================

SUBJECTS_TO_PREPROCESS = [
    "A07",
    "A08",
]


for subject in SUBJECTS_TO_PREPROCESS:

    output_path = (
        PREPARED_DIR
        /
        f"{subject}_GLASS_preprocessed_seed42.npz"
    )


    print("\n" + "=" * 84)
    print("PREPROCESSING", subject)
    print("=" * 84)


    # --------------------------------------------------------
    # Reuse a verified existing output
    # --------------------------------------------------------

    if output_path.is_file():

        try:

            with np.load(
                output_path,
                allow_pickle=False
            ) as existing:

                valid = (
                    "X_glass_processed_grouped"
                    in existing.files
                    and
                    "y_grouped"
                    in existing.files
                    and
                    existing[
                        "X_glass_processed_grouped"
                    ].shape
                    ==
                    (
                        35,10,2,6,8,128
                    )
                    and
                    existing[
                        "y_grouped"
                    ].shape
                    ==
                    (
                        35,10,2,6
                    )
                )

            if valid:

                print(
                    "Verified prepared file already exists."
                )

                print(
                    "Reusing:",
                    output_path
                )

                continue

        except Exception:

            pass


    # --------------------------------------------------------
    # Create
    # --------------------------------------------------------

    result = preprocess_subject(
        subject
    )


    # Atomic save
    temp_path = (
        PREPARED_DIR
        /
        f"{subject}_prepared.tmp"
    )

    with temp_path.open(
        "wb"
    ) as handle:

        np.savez_compressed(
            handle,
            **result
        )

    temp_path.replace(
        output_path
    )


    # --------------------------------------------------------
    # Verify saved artifact
    # --------------------------------------------------------

    with np.load(
        output_path,
        allow_pickle=False
    ) as saved:

        X_saved = np.asarray(
            saved[
                "X_glass_processed_grouped"
            ]
        )

        y_saved = np.asarray(
            saved[
                "y_grouped"
            ]
        )


    assert X_saved.shape == (
        35,10,2,6,8,128
    )

    assert y_saved.shape == (
        35,10,2,6
    )

    assert np.isfinite(
        X_saved
    ).all()

    assert np.all(
        y_saved.sum(axis=-1) == 1
    )


    print(
        "Saved:",
        output_path
    )

    print(
        "X:",
        X_saved.shape
    )

    print(
        "y:",
        y_saved.shape
    )

    print(
        "EEG min/max/mean/std:",
        float(X_saved.min()),
        float(X_saved.max()),
        float(X_saved.mean()),
        float(X_saved.std())
    )

    print(
        "File size:",
        round(
            output_path.stat().st_size
            / 1024**2,
            2
        ),
        "MB"
    )


    save_progress(
        subject=subject,
        stage="preprocessing_complete",
        status="completed"
    )

    del result
    del X_saved
    del y_saved

    gc.collect()


print("\n" + "=" * 84)
print("A07 + A08 PREPROCESSING COMPLETE")
print("=" * 84)

print("Model training performed: NO")
print("Outer-fold performance evaluated: NO")


In [ ]:
# ============================================================
# KAGGLE — INSTALL ORIGINAL EEG-GANET SOURCE PATCH
# ============================================================

from pathlib import Path
import gdown
import shutil

PATCH_FILE_ID = "1Gpq_8OroTlepq5pDPBtwkNdlbMaseZu-"

PATCH_ZIP = Path(
    "/kaggle/working/EEG_GANet_source_patch.zip"
)

PATCH_DEST = (
    PROJECT_ROOT
    / "code"
    / "EEG-GANet"
)

gdown.download(
    id=PATCH_FILE_ID,
    output=str(PATCH_ZIP),
    quiet=False
)

assert PATCH_ZIP.is_file()

shutil.unpack_archive(
    str(PATCH_ZIP),
    str(PATCH_DEST)
)

print("Patch extracted to:")
print(PATCH_DEST)

print(
    "\nGANs.py exists:",
    (PATCH_DEST / "GANs.py").is_file()
)

print(
    "github_model.py exists:",
    (PATCH_DEST / "github_model.py").is_file()
)

print(
    "github_Utils.py exists:",
    (PATCH_DEST / "github_Utils.py").is_file()
)


In [ ]:
# ============================================================
# KAGGLE STEP 4B — EEG-GANET DEPENDENCY / IMPORT AUDIT
# ============================================================

from pathlib import Path
import sys
import hashlib
import importlib

CODE_DIR = (
    PROJECT_ROOT
    / "code"
    / "EEG-GANet"
)

assert CODE_DIR.is_dir(), CODE_DIR

print("CODE_DIR:")
print(CODE_DIR)

# ------------------------------------------------------------
# 1. Critical dependency files
# ------------------------------------------------------------

for filename in [
    "github_model.py",
    "github_Utils.py",
    "GANs.py",
]:

    path = CODE_DIR / filename

    print(
        filename,
        "->",
        "PASS" if path.is_file() else "MISSING"
    )

    assert path.is_file(), path


# ------------------------------------------------------------
# 2. Preserve original source hash
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


EXPECTED_DBNET_HASH = (
    "f3e570d12de34080ea13003a37bf9a2493b93f1a2376631e8224a40942231cd5"
)

actual_hash = sha256_file(
    CODE_DIR / "github_model.py"
)

print("\nDBNet SHA256:")
print(actual_hash)

assert actual_hash == EXPECTED_DBNET_HASH

print("DBNet source hash: PASS")


# ------------------------------------------------------------
# 3. Put source directory on Python path
# ------------------------------------------------------------

if str(CODE_DIR) not in sys.path:

    sys.path.insert(
        0,
        str(CODE_DIR)
    )


# Remove failed partial imports from the previous attempt
for name in [
    "github_model",
    "github_Utils",
    "GANs",
]:

    sys.modules.pop(
        name,
        None
    )


# ------------------------------------------------------------
# 4. Test original import chain
# ------------------------------------------------------------

GANs = importlib.import_module(
    "GANs"
)

github_Utils = importlib.import_module(
    "github_Utils"
)

github_model = importlib.import_module(
    "github_model"
)

assert hasattr(
    github_model,
    "EEG_DBNet_V2"
)

print("\nGANs import: PASS")
print("github_Utils import: PASS")
print("github_model import: PASS")
print("EEG_DBNet_V2 available: PASS")


print("\n" + "=" * 80)
print("EEG-GANET SOURCE DEPENDENCY AUDIT: PASS")
print("=" * 80)


In [ ]:
# ============================================================
# KAGGLE — RESTORE ALL FROZEN MODEL DEFINITIONS
# Run this ONCE after a kernel/runtime restart.
#
# NO TRAINING
# NO OUTER DATA ACCESS
# ============================================================

from pathlib import Path
import sys
import gc
import hashlib
import importlib

import numpy as np
import tensorflow as tf


# ============================================================
# 1. PROJECT ROOT
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/working/EEG_GANet_Reproduction"
)

assert PROJECT_ROOT.is_dir(), PROJECT_ROOT

CV_ROOT = (
    PROJECT_ROOT
    / "GLASS_GANet"
    / "results"
    / "paper_confirmatory_cv_v1"
)


# ============================================================
# 2. DETERMINISM
# ============================================================

try:
    tf.config.experimental.enable_op_determinism()
    print("Deterministic TensorFlow ops: ENABLED")
except Exception as e:
    print("Determinism:", e)


# ============================================================
# 3. FROZEN CONSTANTS
# ============================================================

NUMBER_OF_CHOICES = 6
NUMBER_OF_CHANNELS = 8
NUMBER_OF_SAMPLES = 128
NUMBER_OF_INTERACTIONS = 28

LEARNING_RATE = 0.0009

GROUP_BATCH_SIZE = 32
BINARY_BATCH_SIZE = 64

MAX_EPOCHS = 1000
PATIENCE = 50
MIN_DELTA = 1e-4

EXPECTED_PARAMS = {
    "structured_dbnet": 3961,
    "glass_gated_dbnet": 4986,
    "interaction_dbnet": 3989,
    "binary_dbnet": 4090,
}


# ============================================================
# 4. LOAD ORIGINAL EEG-DBNET SOURCE
# ============================================================

CODE_DIR = (
    PROJECT_ROOT
    / "code"
    / "EEG-GANet"
)

MODEL_FILE = (
    CODE_DIR
    / "github_model.py"
)

UTILS_FILE = (
    CODE_DIR
    / "github_Utils.py"
)

GANS_FILE = (
    CODE_DIR
    / "GANs.py"
)

assert MODEL_FILE.is_file(), MODEL_FILE
assert UTILS_FILE.is_file(), UTILS_FILE
assert GANS_FILE.is_file(), GANS_FILE


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


EXPECTED_DBNET_HASH = (
    "f3e570d12de34080ea13003a37bf9a2493b93f1a2376631e8224a40942231cd5"
)

assert (
    sha256_file(MODEL_FILE)
    ==
    EXPECTED_DBNET_HASH
)

print("DBNet source hash: PASS")


# Put original source directory on Python path
if str(CODE_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(CODE_DIR)
    )


# Clear failed imports from previous attempts
for module_name in [
    "GANs",
    "github_Utils",
    "github_model",
]:
    sys.modules.pop(
        module_name,
        None
    )


# Import in dependency order
GANs = importlib.import_module(
    "GANs"
)

github_Utils = importlib.import_module(
    "github_Utils"
)

github_model = importlib.import_module(
    "github_model"
)

EEG_DBNet_V2 = (
    github_model.EEG_DBNet_V2
)

print("GANs import: PASS")
print("github_Utils import: PASS")
print("github_model import: PASS")


# ============================================================
# 5. BINARY TARGET AUC
# ============================================================

class TargetAUC(
    tf.keras.metrics.AUC
):

    def __init__(
        self,
        name="target_auc",
        **kwargs
    ):

        super().__init__(
            name=name,
            curve="ROC",
            **kwargs
        )


    def update_state(
        self,
        y_true,
        y_pred,
        sample_weight=None
    ):

        return super().update_state(
            y_true[:, 1],
            y_pred[:, 1],
            sample_weight=sample_weight
        )


# ============================================================
# 6. SHARED DBNET EPOCH SCORER
# ============================================================

def build_epoch_scorer():

    public_model = (
        EEG_DBNet_V2(
            NumFilter=8,
            SamplingFrequency=128,
            NumChannels=8,
            FilterScaler=2,
            NumClasses=2,
            DropoutRate=0.5,
        )
        .build_model()
    )


    if not isinstance(
        public_model.layers[-3],
        tf.keras.layers.Concatenate
    ):
        raise RuntimeError(
            "Unexpected DBNet feature-extractor layer."
        )


    feature_extractor = tf.keras.Model(
        inputs=
            public_model.input,

        outputs=
            public_model
            .layers[-3]
            .output,

        name=
            "dbnet_v2_feature_extractor",
    )


    epoch_input = tf.keras.Input(
        shape=(
            8,
            128,
            1
        ),
        name=
            "single_stimulus_epoch"
    )


    features = feature_extractor(
        epoch_input
    )


    score = tf.keras.layers.Dense(
        1,

        kernel_constraint=
            tf.keras.constraints
            .max_norm(
                0.25
            ),

        name=
            "stimulus_score",
    )(features)


    return tf.keras.Model(
        epoch_input,
        score,
        name=
            "dbnet_v2_epoch_scorer"
    )


# ============================================================
# 7. STRUCTURED DBNET
# ============================================================

def build_structured_dbnet():

    scorer = (
        build_epoch_scorer()
    )


    group_input = tf.keras.Input(
        shape=(
            6,
            8,
            128,
            1
        ),
        name=
            "six_stimulus_group"
    )


    scores = (
        tf.keras.layers
        .TimeDistributed(
            scorer,
            name=
                "shared_dbnet_scorer"
        )(group_input)
    )


    logits = (
        tf.keras.layers
        .Reshape(
            (6,),
            name=
                "dbnet_logits"
        )(scores)
    )


    probabilities = (
        tf.keras.layers.Softmax(
            axis=-1,
            name=
                "six_choice_probabilities"
        )(logits)
    )


    model = tf.keras.Model(
        group_input,
        probabilities,
        name=
            "structured_dbnet_v2"
    )


    model.compile(
        optimizer=
            tf.keras.optimizers.Adam(
                learning_rate=
                    LEARNING_RATE
            ),

        loss=
            tf.keras.losses
            .CategoricalCrossentropy(),

        metrics=[
            tf.keras.metrics
            .CategoricalAccuracy(
                name=
                    "six_choice_accuracy"
            )
        ],
    )


    return model


# ============================================================
# 8. FIXED GLASS SCORES
# ============================================================

class FixedGlassScores(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        beta_matrix,
        **kwargs
    ):

        super().__init__(
            trainable=False,
            **kwargs
        )

        beta_matrix = np.asarray(
            beta_matrix,
            dtype=np.float32
        )

        if beta_matrix.shape != (
            8,
            128
        ):
            raise ValueError(
                beta_matrix.shape
            )

        self._initial_beta = (
            beta_matrix
        )


    def build(
        self,
        input_shape
    ):

        self.beta_matrix = (
            self.add_weight(
                name=
                    "beta_matrix",

                shape=(
                    8,
                    128
                ),

                initializer=
                    tf.keras.initializers
                    .Constant(
                        self._initial_beta
                    ),

                trainable=False,
            )
        )

        super().build(
            input_shape
        )


    def call(
        self,
        inputs
    ):

        return tf.einsum(
            "bkct,ct->bk",

            tf.cast(
                inputs[..., 0],
                self.compute_dtype
            ),

            self.beta_matrix
        )


# ============================================================
# 9. NONNEGATIVE GLASS GATE
# ============================================================

class NonnegativeGlassResidual(
    tf.keras.layers.Layer
):

    def build(
        self,
        input_shape
    ):

        self.glass_gate = (
            self.add_weight(
                name=
                    "glass_gate",

                shape=(),

                initializer=
                    "zeros",

                trainable=True,

                constraint=
                    tf.keras.constraints
                    .NonNeg(),
            )
        )

        super().build(
            input_shape
        )


    def call(
        self,
        inputs
    ):

        dbnet_logits, glass_logits = (
            inputs
        )


        centered = (
            glass_logits
            -
            tf.reduce_mean(
                glass_logits,
                axis=-1,
                keepdims=True
            )
        )


        scale = (
            tf.math.reduce_std(
                centered,
                axis=-1,
                keepdims=True
            )
        )


        normalized = (
            centered
            /
            (
                scale
                +
                1e-6
            )
        )


        return (
            dbnet_logits
            +
            self.glass_gate
            *
            normalized
        )


# ============================================================
# 10. GLASS-GATED STRUCTURED DBNET
# ============================================================

def build_gated_hybrid(
    beta_matrix
):

    scorer = (
        build_epoch_scorer()
    )


    group_input = tf.keras.Input(
        shape=(
            6,
            8,
            128,
            1
        ),
        name=
            "six_stimulus_group"
    )


    dbnet_scores = (
        tf.keras.layers
        .TimeDistributed(
            scorer,
            name=
                "shared_dbnet_scorer"
        )(group_input)
    )


    dbnet_logits = (
        tf.keras.layers.Reshape(
            (6,),
            name=
                "dbnet_logits"
        )(dbnet_scores)
    )


    glass_logits = (
        FixedGlassScores(
            beta_matrix,
            name=
                "fixed_official_glass_scores"
        )(group_input)
    )


    combined = (
        NonnegativeGlassResidual(
            name=
                "nonnegative_glass_residual"
        )(
            [
                dbnet_logits,
                glass_logits,
            ]
        )
    )


    probabilities = (
        tf.keras.layers.Softmax(
            axis=-1,
            name=
                "six_choice_probabilities"
        )(combined)
    )


    model = tf.keras.Model(
        group_input,
        probabilities,
        name=
            "glass_gated_structured_dbnet"
    )


    model.compile(
        optimizer=
            tf.keras.optimizers.Adam(
                learning_rate=
                    LEARNING_RATE
            ),

        loss=
            tf.keras.losses
            .CategoricalCrossentropy(),

        metrics=[
            tf.keras.metrics
            .CategoricalAccuracy(
                name=
                    "six_choice_accuracy"
            )
        ],
    )


    return model


# ============================================================
# 11. FISHER-Z INTERACTION FEATURES
# ============================================================

CHANNEL_PAIRS = np.asarray(
    [
        (u, v)
        for u in range(7)
        for v in range(
            u + 1,
            8
        )
    ],
    dtype=np.int64
)

assert CHANNEL_PAIRS.shape == (
    28,
    2
)


def compute_fisher_features(
    grouped_eeg
):

    eeg = np.asarray(
        grouped_eeg,
        dtype=np.float64
    )[..., 0]


    groups = eeg.shape[0]


    flat = eeg.reshape(
        groups * 6,
        8,
        128
    )


    if not np.all(
        np.std(
            flat,
            axis=-1
        ) > 0
    ):
        raise RuntimeError(
            "Zero-variance EEG epoch."
        )


    features = np.empty(
        (
            flat.shape[0],
            28
        ),
        dtype=np.float64
    )


    for idx, (
        a,
        b
    ) in enumerate(
        CHANNEL_PAIRS
    ):

        x = flat[:, a, :]
        z = flat[:, b, :]


        xc = (
            x
            -
            x.mean(
                axis=1,
                keepdims=True
            )
        )

        zc = (
            z
            -
            z.mean(
                axis=1,
                keepdims=True
            )
        )


        r = (
            (xc * zc).sum(
                axis=1
            )
            /
            np.sqrt(
                (xc ** 2).sum(
                    axis=1
                )
                *
                (zc ** 2).sum(
                    axis=1
                )
            )
        )


        features[:, idx] = (
            np.arctanh(
                np.clip(
                    r,
                    -0.999999,
                    0.999999
                )
            )
        )


    return (
        features
        .reshape(
            groups,
            6,
            28
        )
        .astype(
            np.float32
        )
    )


# ============================================================
# 12. INTERACTION DBNET
# ============================================================

def build_interaction_dbnet():

    scorer = (
        build_epoch_scorer()
    )


    eeg_input = tf.keras.Input(
        shape=(
            6,
            8,
            128,
            1
        ),
        name=
            "six_stimulus_eeg"
    )


    interaction_input = (
        tf.keras.Input(
            shape=(
                6,
                28
            ),
            name=
                "six_stimulus_rtgp_interactions"
        )
    )


    dbnet_scores = (
        tf.keras.layers
        .TimeDistributed(
            scorer,
            name=
                "shared_dbnet_scorer"
        )(eeg_input)
    )


    dbnet_logits = (
        tf.keras.layers.Reshape(
            (6,),
            name=
                "dbnet_logits"
        )(dbnet_scores)
    )


    interaction_scores = (
        tf.keras.layers
        .TimeDistributed(

            tf.keras.layers.Dense(
                1,

                use_bias=False,

                kernel_initializer=
                    "zeros",

                name=
                    "interaction_linear"
            ),

            name=
                "shared_rtgp_interaction_scorer"

        )(interaction_input)
    )


    interaction_logits = (
        tf.keras.layers.Reshape(
            (6,),
            name=
                "interaction_logits_unscaled"
        )(interaction_scores)
    )


    interaction_logits = (
        tf.keras.layers.Rescaling(
            scale=
                1.0
                /
                28.0,

            name=
                "rtgp_ns2_scaling"
        )(interaction_logits)
    )


    combined = (
        tf.keras.layers.Add(
            name=
                "combined_choice_logits"
        )(
            [
                dbnet_logits,
                interaction_logits
            ]
        )
    )


    probabilities = (
        tf.keras.layers.Softmax(
            axis=-1,
            name=
                "six_choice_probabilities"
        )(combined)
    )


    model = tf.keras.Model(
        [
            eeg_input,
            interaction_input
        ],
        probabilities,
        name=
            "structured_dbnet_plus_rtgp_interactions"
    )


    model.compile(
        optimizer=
            tf.keras.optimizers.Adam(
                learning_rate=
                    LEARNING_RATE
            ),

        loss=
            tf.keras.losses
            .CategoricalCrossentropy(),

        metrics=[
            tf.keras.metrics
            .CategoricalAccuracy(
                name=
                    "six_choice_accuracy"
            )
        ],
    )


    return model


# ============================================================
# 13. BINARY DBNET
# ============================================================

def build_binary_dbnet_frozen():

    model = (
        EEG_DBNet_V2(
            NumFilter=8,
            SamplingFrequency=128,
            NumChannels=8,
            FilterScaler=2,
            NumClasses=2,
            DropoutRate=0.5,
        )
        .build_model()
    )


    model.compile(
        optimizer=
            tf.keras.optimizers.Adam(
                learning_rate=
                    LEARNING_RATE
            ),

        loss=
            tf.keras.losses
            .CategoricalCrossentropy(),

        metrics=[
            TargetAUC(
                name=
                    "target_auc"
            )
        ],
    )


    return model


# ============================================================
# 14. PARAMETER AUDIT
# ============================================================

dummy_beta = np.zeros(
    (
        8,
        128
    ),
    dtype=np.float32
)


models_to_test = {}


tf.keras.backend.clear_session()
gc.collect()
tf.keras.utils.set_random_seed(1)

models_to_test[
    "structured_dbnet"
] = build_structured_dbnet()


tf.keras.backend.clear_session()
gc.collect()
tf.keras.utils.set_random_seed(1)

models_to_test[
    "glass_gated_dbnet"
] = build_gated_hybrid(
    dummy_beta
)


tf.keras.backend.clear_session()
gc.collect()
tf.keras.utils.set_random_seed(1)

models_to_test[
    "interaction_dbnet"
] = build_interaction_dbnet()


tf.keras.backend.clear_session()
gc.collect()
tf.keras.utils.set_random_seed(1)

models_to_test[
    "binary_dbnet"
] = build_binary_dbnet_frozen()


print("\n" + "=" * 80)
print("FROZEN PARAMETER AUDIT")
print("=" * 80)


for name, model in (
    models_to_test.items()
):

    count = int(
        model.count_params()
    )

    print(
        f"{name:24s} "
        f"actual={count} "
        f"expected={EXPECTED_PARAMS[name]}"
    )

    assert (
        count
        ==
        EXPECTED_PARAMS[
            name
        ]
    )


# Gate starts exactly at zero
gate = (
    models_to_test[
        "glass_gated_dbnet"
    ]
    .get_layer(
        "nonnegative_glass_residual"
    )
    .glass_gate
    .numpy()
)

assert float(gate) == 0.0


# Interaction starts exactly at zero
interaction_kernel = (
    models_to_test[
        "interaction_dbnet"
    ]
    .get_layer(
        "shared_rtgp_interaction_scorer"
    )
    .layer
    .get_weights()[0]
)

assert interaction_kernel.shape == (
    28,
    1
)

assert np.count_nonzero(
    interaction_kernel
) == 0


del models_to_test

tf.keras.backend.clear_session()
gc.collect()


# ============================================================
# 15. FINAL FUNCTION CHECK
# ============================================================

required = [
    "build_structured_dbnet",
    "build_gated_hybrid",
    "build_interaction_dbnet",
    "build_binary_dbnet_frozen",
    "compute_fisher_features",
]

print("\nFUNCTION AVAILABILITY")

for name in required:

    print(
        f"{name:35s}",
        "READY"
        if name in globals()
        else "MISSING"
    )

    assert name in globals()


print("\n" + "=" * 80)
print("KAGGLE FROZEN MODEL AUDIT: PASS")
print("=" * 80)

print("Training performed: NO")
print("Outer-test EEG accessed: NO")
print("Outer performance evaluated: NO")


In [ ]:
# ============================================================
# KAGGLE — FINAL PRE-RUN CHECK
# ============================================================

from pathlib import Path
import time
import tensorflow as tf

# Start unattended-training timer NOW.
SESSION_START = time.monotonic()
SAFE_LIMIT_HOURS = 10.5

SUBJECT_QUEUE = [
    "A07",
    "A08",
]

for subject in SUBJECT_QUEUE:

    prepared = (
        PROJECT_ROOT
        / "GLASS_GANet"
        / "prepared_data"
        / f"{subject}_GLASS_preprocessed_seed42.npz"
    )

    assert prepared.is_file(), prepared

    print(
        subject,
        "prepared data: PASS"
    )


required = [
    "build_structured_dbnet",
    "build_gated_hybrid",
    "build_interaction_dbnet",
    "build_binary_dbnet_frozen",
    "compute_fisher_features",
]

for name in required:
    assert name in globals(), name

print("\nFrozen model functions: PASS")
print("TensorFlow:", tf.__version__)
print(
    "GPU:",
    tf.config.list_physical_devices("GPU")
)

print("\nSafe unattended budget: 10.5 hours")
print("Outer-test evaluation: DISABLED")

print("\nREADY FOR UNATTENDED TRAINING")


In [ ]:
# ============================================================
# KAGGLE STEP 6 — UNATTENDED RESTART-SAFE TRAINING RUNNER
#
# Queue:
#   A07 Fold 1 -> 7
#   A08 Fold 1 -> 7 (as time allows)
#
# For each fold:
#   1. train/inner-val data only
#   2. fold-local GLASS Stage 1
#   3. fold-local GLASS Stage 2
#   4. Structured/Gated/Interaction, seeds 1,2,3
#   5. Binary DBNet, seeds 1,2,3
#   6. checkpoint-load audit
#   7. recovery ZIP
#
# IMPORTANT:
#   OUTER EEG/LABELS ARE NEVER ACCESSED.
#   OUTER PERFORMANCE IS NEVER EVALUATED.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import gc
import json
import time
import shutil
import importlib.util
import hashlib

import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# 0. REQUIRED DEFINITIONS AUDIT
# ============================================================

REQUIRED_OBJECTS = [
    "PROJECT_ROOT",
    "CV_ROOT",
    "FOLDS",
    "build_structured_dbnet",
    "build_gated_hybrid",
    "build_interaction_dbnet",
    "build_binary_dbnet_frozen",
    "compute_fisher_features",
]

for name in REQUIRED_OBJECTS:
    if name not in globals():
        raise RuntimeError(
            f"Missing required object: {name}\n"
            "Rerun the previous setup/model-definition cells."
        )

print("Required frozen definitions: PASS")


# ============================================================
# 1. SESSION / TIME GUARD
# ============================================================

# Reuse Step-3 timer if still present.
if "SESSION_START" not in globals():
    SESSION_START = time.monotonic()

if "SAFE_LIMIT_HOURS" not in globals():
    SAFE_LIMIT_HOURS = 10.5


def elapsed_hours():
    return (
        time.monotonic()
        - SESSION_START
    ) / 3600.0


def remaining_safe_hours():
    return (
        SAFE_LIMIT_HOURS
        - elapsed_hours()
    )


# Don't begin another substantial operation once inside this
# reserve. There is still extra margin before Kaggle's hard cap.
MIN_REMAINING_HOURS = 0.35


# ============================================================
# 2. FROZEN SETTINGS
# ============================================================

SUBJECT_QUEUE = [
    "A07",
    "A08",
]

SEEDS = [
    1,
    2,
    3,
]

LEARNING_RATE = 0.0009

MAX_EPOCHS = 1000
PATIENCE = 50
MIN_DELTA = 1e-4

GROUP_BATCH_SIZE = 32
BINARY_BATCH_SIZE = 64

EXPECTED_PARAMS = {
    "structured_dbnet": 3961,
    "glass_gated_dbnet": 4986,
    "interaction_dbnet": 3989,
    "binary_dbnet": 4090,
}


# Frozen structured-family order used in the A02 runner.
STRUCTURED_ORDER = {

    1: [
        "structured_dbnet",
        "glass_gated_dbnet",
        "interaction_dbnet",
    ],

    2: [
        "interaction_dbnet",
        "glass_gated_dbnet",
        "structured_dbnet",
    ],

    3: [
        "structured_dbnet",
        "glass_gated_dbnet",
        "interaction_dbnet",
    ],
}


# Frozen GLASS settings
GLASS_NUM_STEPS = 2000
GLASS_SAMPLE_SIZE = 10
GLASS_IMPORTANCE_SAMPLE_SIZE = 10
GLASS_POSTERIOR_SAMPLE_SIZE = 5000
GLASS_LEARNING_RATE = 0.05
GLASS_SEED = 1


# ============================================================
# 3. RESULT / RECOVERY PATHS
# ============================================================

SUBJECT_RUN_ROOT = (
    CV_ROOT
    / "subject_runs"
)

SUBJECT_RUN_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


RECOVERY_ROOT = Path(
    "/kaggle/working/recovery"
)

RECOVERY_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


PROGRESS_FILE = (
    RECOVERY_ROOT
    / "progress_manifest.json"
)


# ============================================================
# 4. PROGRESS WRITER
# ============================================================

def save_progress(
    subject=None,
    fold=None,
    stage=None,
    seed=None,
    model=None,
    status="running",
):

    payload = {
        "subject": subject,
        "fold": fold,
        "stage": stage,
        "seed": seed,
        "model": model,
        "status": status,

        "elapsed_hours":
            elapsed_hours(),

        "remaining_safe_hours":
            remaining_safe_hours(),

        "safe_limit_hours":
            SAFE_LIMIT_HOURS,

        "updated_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "outer_test_eeg_accessed":
            False,

        "outer_performance_evaluated":
            False,
    }


    tmp = (
        RECOVERY_ROOT
        / "progress_manifest.tmp"
    )

    tmp.write_text(
        json.dumps(
            payload,
            indent=2,
            allow_nan=False
        ),
        encoding="utf-8"
    )

    tmp.replace(
        PROGRESS_FILE
    )


# ============================================================
# 5. ATOMIC JSON
# ============================================================

def atomic_json_save(
    path,
    payload
):

    path = Path(path)

    tmp = (
        path.parent
        / (
            path.stem
            + ".tmp"
        )
    )

    tmp.write_text(
        json.dumps(
            payload,
            indent=2,
            allow_nan=False
        ),
        encoding="utf-8"
    )

    tmp.replace(
        path
    )


# ============================================================
# 6. SNAPSHOT CURRENT FOLD
# ============================================================

def snapshot_fold(
    subject,
    fold,
    suffix="snapshot",
):

    fold_root = (
        SUBJECT_RUN_ROOT
        / subject
        / f"outer_fold_{fold}"
    )

    if not fold_root.exists():
        return None


    archive_base = (
        RECOVERY_ROOT
        / (
            f"{subject}_outer_fold_{fold}_"
            f"{suffix}"
        )
    )

    archive_path = Path(
        str(archive_base)
        + ".zip"
    )


    if archive_path.exists():
        archive_path.unlink()


    result = shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=str(fold_root)
    )

    return Path(result)


# ============================================================
# 7. SAFE STOP
# ============================================================

def time_guard(
    subject,
    fold,
    stage,
    seed=None,
    model=None,
):

    remaining = (
        remaining_safe_hours()
    )

    if remaining > MIN_REMAINING_HOURS:
        return


    save_progress(
        subject=subject,
        fold=fold,
        stage=stage,
        seed=seed,
        model=model,
        status="safe_stop",
    )


    snap = snapshot_fold(
        subject,
        fold,
        suffix="safe_stop"
    )


    print("\n" + "=" * 90)
    print("SAFE SESSION STOP")
    print("=" * 90)

    print(
        "Elapsed:",
        round(
            elapsed_hours(),
            2
        ),
        "hours"
    )

    print(
        "Remaining safe budget:",
        round(
            remaining,
            2
        ),
        "hours"
    )

    print(
        "Current subject/fold:",
        subject,
        fold
    )

    print(
        "Current stage:",
        stage
    )

    print(
        "Recovery snapshot:",
        snap
    )

    print(
        "\nCompleted files already in "
        "/kaggle/working are retained."
    )

    raise SystemExit(
        "Stopped safely before session limit."
    )


# ============================================================
# 8. PARTIAL-DIRECTORY HANDLING
# ============================================================

def quarantine_partial(
    run_dir
):

    run_dir = Path(
        run_dir
    )

    if not run_dir.exists():
        return


    stamp = datetime.now(
        timezone.utc
    ).strftime(
        "%Y%m%dT%H%M%SZ"
    )

    quarantine = (
        run_dir.parent
        / (
            run_dir.name
            + ".partial_"
            + stamp
        )
    )

    run_dir.rename(
        quarantine
    )

    print(
        "  Partial run quarantined:",
        quarantine.name
    )


# ============================================================
# 9. MODEL COMPLETION CHECK
# ============================================================

def model_complete(
    run_dir,
    subject,
    fold,
    seed,
    model_name
):

    run_dir = Path(
        run_dir
    )

    weights = (
        run_dir
        / "best.weights.h5"
    )

    summary = (
        run_dir
        / "training_summary.json"
    )


    if not (
        weights.is_file()
        and
        summary.is_file()
    ):
        return False


    try:

        payload = json.loads(
            summary.read_text(
                encoding="utf-8"
            )
        )

        return (
            payload.get(
                "status"
            ) == "completed"

            and
            payload.get(
                "subject"
            ) == subject

            and
            int(
                payload.get(
                    "outer_fold"
                )
            ) == int(fold)

            and
            int(
                payload.get(
                    "seed"
                )
            ) == int(seed)

            and
            payload.get(
                "model"
            ) == model_name

            and
            int(
                payload.get(
                    "parameter_count"
                )
            )
            ==
            EXPECTED_PARAMS[
                model_name
            ]

            and
            payload.get(
                "outer_test_used_for_training"
            ) is False
        )

    except Exception:
        return False


# ============================================================
# 10. IMPORT GLASS TF2.20
# ============================================================

GLASS_TF220_FILE = (
    PROJECT_ROOT
    / "GLASS_GANet"
    / "integration"
    / "glass_tf220.py"
)

assert GLASS_TF220_FILE.is_file(), (
    GLASS_TF220_FILE
)


spec = (
    importlib.util.spec_from_file_location(
        "glass_tf220_unattended",
        GLASS_TF220_FILE
    )
)

glass_module = (
    importlib.util.module_from_spec(
        spec
    )
)

spec.loader.exec_module(
    glass_module
)

Glass = glass_module.Glass

print("GLASS TF2.20 import: PASS")


# ============================================================
# 11. LOAD ONE PREPARED SUBJECT
# ============================================================

def load_subject(
    subject
):

    path = (
        PROJECT_ROOT
        / "GLASS_GANet"
        / "prepared_data"
        / (
            f"{subject}_"
            "GLASS_preprocessed_seed42.npz"
        )
    )

    assert path.is_file(), (
        f"Missing prepared data: {path}"
    )


    with np.load(
        path,
        allow_pickle=False
    ) as archive:

        X = np.asarray(
            archive[
                "X_glass_processed_grouped"
            ],
            dtype=np.float32
        )

        y = np.asarray(
            archive[
                "y_grouped"
            ],
            dtype=np.float32
        )


    assert X.shape == (
        35,
        10,
        2,
        6,
        8,
        128
    )

    assert y.shape == (
        35,
        10,
        2,
        6
    )

    assert np.isfinite(
        X
    ).all()

    assert np.isfinite(
        y
    ).all()


    return X, y


# ============================================================
# 12. PREPARE TRAIN + INNER VALIDATION ONLY
# ============================================================

def prepare_fold_data(
    X_all,
    y_all,
    fold
):

    split = FOLDS[
        fold
    ]


    train_chars = list(
        split["train"]
    )

    val_chars = list(
        split["val"]
    )

    # IMPORTANT:
    # outer character indices are known,
    # but X_all is NEVER indexed using them.
    outer_chars = list(
        split["outer"]
    )


    train_idx = (
        np.asarray(
            train_chars,
            dtype=np.int64
        )
        - 1
    )

    val_idx = (
        np.asarray(
            val_chars,
            dtype=np.int64
        )
        - 1
    )


    X_train_char = np.asarray(
        X_all[
            train_idx
        ],
        dtype=np.float32
    )

    y_train_char = np.asarray(
        y_all[
            train_idx
        ],
        dtype=np.float32
    )


    X_val_char = np.asarray(
        X_all[
            val_idx
        ],
        dtype=np.float32
    )

    y_val_char = np.asarray(
        y_all[
            val_idx
        ],
        dtype=np.float32
    )


    X_train = (
        X_train_char
        .reshape(
            500,
            6,
            8,
            128
        )
        [..., np.newaxis]
    )

    y_train = (
        y_train_char
        .reshape(
            500,
            6
        )
    )


    X_val = (
        X_val_char
        .reshape(
            100,
            6,
            8,
            128
        )
        [..., np.newaxis]
    )

    y_val = (
        y_val_char
        .reshape(
            100,
            6
        )
    )


    assert X_train.shape == (
        500,
        6,
        8,
        128,
        1
    )

    assert y_train.shape == (
        500,
        6
    )

    assert X_val.shape == (
        100,
        6,
        8,
        128,
        1
    )

    assert y_val.shape == (
        100,
        6
    )


    np.testing.assert_array_equal(
        y_train.sum(
            axis=1
        ),
        np.ones(500)
    )

    np.testing.assert_array_equal(
        y_val.sum(
            axis=1
        ),
        np.ones(100)
    )


    return {
        "train_chars":
            train_chars,

        "val_chars":
            val_chars,

        "outer_chars":
            outer_chars,

        "X_train":
            X_train,

        "y_train":
            y_train,

        "X_val":
            X_val,

        "y_val":
            y_val,
    }


# ============================================================
# 13. GLASS FIT — RESTART SAFE
# ============================================================

def run_glass(
    subject,
    fold,
    fold_data
):

    fold_root = (
        SUBJECT_RUN_ROOT
        / subject
        / f"outer_fold_{fold}"
    )

    glass_dir = (
        fold_root
        / "glass"
    )

    glass_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    stage1_path = (
        glass_dir
        / "stage1_no_shrinkage.npz"
    )

    stage2_path = (
        glass_dir
        / "stage2_shrinkage.npz"
    )


    X_train_glass = np.asarray(
        fold_data[
            "X_train"
        ][..., 0],
        dtype=np.float32
    )

    y_train_glass = np.asarray(
        fold_data[
            "y_train"
        ],
        dtype=np.float32
    )


    train_chars = (
        fold_data[
            "train_chars"
        ]
    )


    # --------------------------------------------------------
    # Completed Stage 2?
    # --------------------------------------------------------

    if stage2_path.is_file():

        try:

            with np.load(
                stage2_path,
                allow_pickle=False
            ) as archive:

                beta = np.asarray(
                    archive[
                        "betaMat"
                    ],
                    dtype=np.float32
                )

                saved_train = np.asarray(
                    archive[
                        "training_characters_1based"
                    ],
                    dtype=np.int64
                )

                used_val = bool(
                    archive[
                        "inner_validation_used_for_fitting"
                    ]
                )

                used_outer = bool(
                    archive[
                        "outer_test_used_for_fitting"
                    ]
                )


            np.testing.assert_array_equal(
                saved_train,
                np.asarray(
                    train_chars,
                    dtype=np.int64
                )
            )

            assert beta.shape == (
                8,
                128
            )

            assert np.isfinite(
                beta
            ).all()

            assert used_val is False
            assert used_outer is False


            print(
                "  GLASS Stage 2: REUSED"
            )

            return beta

        except Exception:

            corrupt = (
                glass_dir
                / (
                    "stage2_invalid_"
                    +
                    datetime.now(
                        timezone.utc
                    ).strftime(
                        "%Y%m%dT%H%M%SZ"
                    )
                    +
                    ".npz"
                )
            )

            stage2_path.rename(
                corrupt
            )

            print(
                "  Invalid Stage-2 quarantined."
            )


    # --------------------------------------------------------
    # Stage 1
    # --------------------------------------------------------

    if stage1_path.is_file():

        with np.load(
            stage1_path,
            allow_pickle=False
        ) as archive:

            beta_stage1 = np.asarray(
                archive[
                    "betaMat"
                ],
                dtype=np.float32
            )

            losses_stage1 = np.asarray(
                archive[
                    "losses"
                ],
                dtype=np.float32
            )

            stage1_seconds = float(
                archive[
                    "stage1_fitting_seconds"
                ]
            )

            saved_train = np.asarray(
                archive[
                    "training_characters_1based"
                ],
                dtype=np.int64
            )


        np.testing.assert_array_equal(
            saved_train,
            np.asarray(
                train_chars,
                dtype=np.int64
            )
        )

        assert beta_stage1.shape == (
            8,
            128
        )

        print(
            "  GLASS Stage 1: REUSED"
        )

    else:

        time_guard(
            subject,
            fold,
            "glass_stage1"
        )

        save_progress(
            subject=subject,
            fold=fold,
            stage="glass_stage1",
            status="running"
        )


        print(
            "  GLASS Stage 1: TRAINING..."
        )


        tf.keras.utils.set_random_seed(
            GLASS_SEED
        )


        glass1 = Glass(
            shrinkage_factor=0.0,
            dtype=tf.float32
        )


        glass1.process_data(
            X_train_glass,
            y_train_glass
        )


        t0 = time.perf_counter()


        with tf.device("/CPU:0"):

            glass1.mfvb(
                num_steps=
                    GLASS_NUM_STEPS,

                sample_size=
                    GLASS_SAMPLE_SIZE,

                importance_sample_size=
                    GLASS_IMPORTANCE_SAMPLE_SIZE,

                learning_rate=
                    GLASS_LEARNING_RATE,

                seed=
                    GLASS_SEED,

                posterior_sample_size=
                    GLASS_POSTERIOR_SAMPLE_SIZE,
            )


        stage1_seconds = (
            time.perf_counter()
            -
            t0
        )


        beta_stage1 = np.asarray(
            glass1.betaMat,
            dtype=np.float32
        )

        losses_stage1 = np.asarray(
            glass1.losses,
            dtype=np.float32
        )


        assert beta_stage1.shape == (
            8,
            128
        )

        assert np.isfinite(
            beta_stage1
        ).all()


        cutoff = float(
            np.median(
                np.abs(
                    beta_stage1
                )
            )
        )


        tmp = (
            glass_dir
            / "stage1_no_shrinkage.tmp"
        )


        with tmp.open(
            "wb"
        ) as handle:

            np.savez_compressed(

                handle,

                betaMat=
                    beta_stage1,

                losses=
                    losses_stage1,

                cutoff=
                    np.asarray(
                        cutoff
                    ),

                shrinkage_factor=
                    np.asarray(
                        0.0
                    ),

                stage1_fitting_seconds=
                    np.asarray(
                        stage1_seconds
                    ),

                training_characters_1based=
                    np.asarray(
                        train_chars,
                        dtype=np.int64
                    ),

                training_data_only_for_fit=
                    np.asarray(
                        True
                    ),

                inner_validation_used_for_fitting=
                    np.asarray(
                        False
                    ),

                outer_test_used_for_fitting=
                    np.asarray(
                        False
                    ),

                num_steps_per_stage=
                    np.asarray(
                        GLASS_NUM_STEPS
                    ),

                sample_size=
                    np.asarray(
                        GLASS_SAMPLE_SIZE
                    ),

                importance_sample_size=
                    np.asarray(
                        GLASS_IMPORTANCE_SAMPLE_SIZE
                    ),

                posterior_sample_size=
                    np.asarray(
                        GLASS_POSTERIOR_SAMPLE_SIZE
                    ),

                learning_rate=
                    np.asarray(
                        GLASS_LEARNING_RATE
                    ),

                random_seed=
                    np.asarray(
                        GLASS_SEED
                    ),
            )


        tmp.replace(
            stage1_path
        )


        print(
            "    Stage 1:",
            round(
                stage1_seconds / 60,
                2
            ),
            "min"
        )


        del glass1
        gc.collect()
        tf.keras.backend.clear_session()


    # --------------------------------------------------------
    # Frozen Stage-2 shrinkage
    # --------------------------------------------------------

    cutoff = float(
        np.median(
            np.abs(
                beta_stage1
            )
        )
    )

    shrinkage = float(
        0.5
        *
        cutoff
    )


    # --------------------------------------------------------
    # Stage 2
    # --------------------------------------------------------

    time_guard(
        subject,
        fold,
        "glass_stage2"
    )


    save_progress(
        subject=subject,
        fold=fold,
        stage="glass_stage2",
        status="running"
    )


    print(
        "  GLASS Stage 2: TRAINING..."
    )

    print(
        "    shrinkage:",
        shrinkage
    )


    tf.keras.utils.set_random_seed(
        GLASS_SEED
    )


    glass2 = Glass(
        shrinkage_factor=
            shrinkage,
        dtype=tf.float32
    )


    glass2.process_data(
        X_train_glass,
        y_train_glass
    )


    t0 = time.perf_counter()


    with tf.device("/CPU:0"):

        glass2.mfvb(
            num_steps=
                GLASS_NUM_STEPS,

            sample_size=
                GLASS_SAMPLE_SIZE,

            importance_sample_size=
                GLASS_IMPORTANCE_SAMPLE_SIZE,

            learning_rate=
                GLASS_LEARNING_RATE,

            seed=
                GLASS_SEED,

            posterior_sample_size=
                GLASS_POSTERIOR_SAMPLE_SIZE,
        )


    stage2_seconds = (
        time.perf_counter()
        -
        t0
    )


    beta_stage2 = np.asarray(
        glass2.betaMat,
        dtype=np.float32
    )

    losses_stage2 = np.asarray(
        glass2.losses,
        dtype=np.float32
    )


    assert beta_stage2.shape == (
        8,
        128
    )

    assert np.isfinite(
        beta_stage2
    ).all()


    total_seconds = (
        stage1_seconds
        +
        stage2_seconds
    )


    tmp = (
        glass_dir
        / "stage2_shrinkage.tmp"
    )


    with tmp.open(
        "wb"
    ) as handle:

        np.savez_compressed(

            handle,

            betaMat=
                beta_stage2,

            losses=
                losses_stage2,

            cutoff=
                np.asarray(
                    cutoff
                ),

            shrinkage_factor=
                np.asarray(
                    shrinkage
                ),

            stage1_fitting_seconds=
                np.asarray(
                    stage1_seconds
                ),

            stage2_fitting_seconds=
                np.asarray(
                    stage2_seconds
                ),

            total_fitting_seconds=
                np.asarray(
                    total_seconds
                ),

            training_characters_1based=
                np.asarray(
                    train_chars,
                    dtype=np.int64
                ),

            training_data_only_for_fit=
                np.asarray(
                    True
                ),

            inner_validation_used_for_fitting=
                np.asarray(
                    False
                ),

            outer_test_used_for_fitting=
                np.asarray(
                    False
                ),

            num_steps_per_stage=
                np.asarray(
                    GLASS_NUM_STEPS
                ),

            sample_size=
                np.asarray(
                    GLASS_SAMPLE_SIZE
                ),

            importance_sample_size=
                np.asarray(
                    GLASS_IMPORTANCE_SAMPLE_SIZE
                ),

            posterior_sample_size=
                np.asarray(
                    GLASS_POSTERIOR_SAMPLE_SIZE
                ),

            learning_rate=
                np.asarray(
                    GLASS_LEARNING_RATE
                ),

            random_seed=
                np.asarray(
                    GLASS_SEED
                ),
        )


    tmp.replace(
        stage2_path
    )


    print(
        "    Stage 2:",
        round(
            stage2_seconds / 60,
            2
        ),
        "min"
    )


    print(
        "    Total GLASS:",
        round(
            total_seconds / 60,
            2
        ),
        "min"
    )


    del glass2
    gc.collect()
    tf.keras.backend.clear_session()


    save_progress(
        subject=subject,
        fold=fold,
        stage="glass_complete",
        status="completed"
    )


    return beta_stage2


# ============================================================
# 14. BUILD MODEL BY NAME
# ============================================================

def construct_model(
    model_name,
    beta
):

    if model_name == "structured_dbnet":

        return (
            build_structured_dbnet()
        )


    if model_name == "glass_gated_dbnet":

        return (
            build_gated_hybrid(
                beta
            )
        )


    if model_name == "interaction_dbnet":

        return (
            build_interaction_dbnet()
        )


    if model_name == "binary_dbnet":

        return (
            build_binary_dbnet_frozen()
        )


    raise ValueError(
        model_name
    )


# ============================================================
# 15. TRAIN ONE NEURAL MODEL
# ============================================================

def train_one_model(
    subject,
    fold,
    seed,
    model_name,
    beta,
    fold_data,
    fisher_train,
    fisher_val,
    X_binary_train,
    y_binary_train_onehot,
    X_binary_val,
    y_binary_val_onehot,
):

    fold_root = (
        SUBJECT_RUN_ROOT
        / subject
        / f"outer_fold_{fold}"
    )


    run_dir = (
        fold_root
        / f"seed_{seed}"
        / model_name
    )


    # --------------------------------------------------------
    # Reuse a verified completed model
    # --------------------------------------------------------

    if model_complete(
        run_dir,
        subject,
        fold,
        seed,
        model_name
    ):

        print(
            f"  {model_name} seed {seed}: REUSED"
        )

        return


    # Existing partial/invalid run -> quarantine
    if run_dir.exists():

        quarantine_partial(
            run_dir
        )


    run_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    time_guard(
        subject,
        fold,
        "neural_training",
        seed,
        model_name
    )


    save_progress(
        subject=subject,
        fold=fold,
        stage="neural_training",
        seed=seed,
        model=model_name,
        status="running"
    )


    # --------------------------------------------------------
    # Deterministic seed
    # --------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()

    tf.keras.utils.set_random_seed(
        seed
    )


    # --------------------------------------------------------
    # Build
    # --------------------------------------------------------

    model = construct_model(
        model_name,
        beta
    )


    assert (
        model.count_params()
        ==
        EXPECTED_PARAMS[
            model_name
        ]
    )


    checkpoint_path = (
        run_dir
        / "best.weights.h5"
    )


    # --------------------------------------------------------
    # Frozen monitor
    # --------------------------------------------------------

    if model_name == "binary_dbnet":

        monitor = (
            "val_target_auc"
        )

        mode = "max"

    else:

        monitor = (
            "val_loss"
        )

        mode = "min"


    callbacks = [

        tf.keras.callbacks.ModelCheckpoint(
            filepath=
                str(
                    checkpoint_path
                ),

            monitor=
                monitor,

            mode=
                mode,

            save_best_only=
                True,

            save_weights_only=
                True,

            verbose=0,
        ),


        tf.keras.callbacks.EarlyStopping(
            monitor=
                monitor,

            mode=
                mode,

            patience=
                PATIENCE,

            min_delta=
                MIN_DELTA,

            restore_best_weights=
                False,

            verbose=0,
        ),
    ]


    # --------------------------------------------------------
    # Fit
    # --------------------------------------------------------

    start = time.perf_counter()


    if model_name == "interaction_dbnet":

        history = model.fit(

            [
                fold_data[
                    "X_train"
                ],
                fisher_train,
            ],

            fold_data[
                "y_train"
            ],

            validation_data=(

                [
                    fold_data[
                        "X_val"
                    ],
                    fisher_val,
                ],

                fold_data[
                    "y_val"
                ],
            ),

            epochs=
                MAX_EPOCHS,

            batch_size=
                GROUP_BATCH_SIZE,

            shuffle=True,

            callbacks=
                callbacks,

            verbose=0,
        )


    elif model_name == "binary_dbnet":

        history = model.fit(

            X_binary_train,

            y_binary_train_onehot,

            validation_data=(
                X_binary_val,
                y_binary_val_onehot,
            ),

            epochs=
                MAX_EPOCHS,

            batch_size=
                BINARY_BATCH_SIZE,

            shuffle=True,

            callbacks=
                callbacks,

            verbose=0,
        )


    else:

        history = model.fit(

            fold_data[
                "X_train"
            ],

            fold_data[
                "y_train"
            ],

            validation_data=(

                fold_data[
                    "X_val"
                ],

                fold_data[
                    "y_val"
                ],
            ),

            epochs=
                MAX_EPOCHS,

            batch_size=
                GROUP_BATCH_SIZE,

            shuffle=True,

            callbacks=
                callbacks,

            verbose=0,
        )


    runtime_seconds = (
        time.perf_counter()
        -
        start
    )


    if not checkpoint_path.is_file():

        raise RuntimeError(
            f"No best checkpoint saved: "
            f"{subject} fold {fold} "
            f"seed {seed} {model_name}"
        )


    # Load best weights before validation artifact save.
    model.load_weights(
        checkpoint_path
    )


    # --------------------------------------------------------
    # Best epoch/value
    # --------------------------------------------------------

    values = np.asarray(
        history.history[
            monitor
        ],
        dtype=np.float64
    )


    if mode == "min":

        best_index = int(
            np.argmin(
                values
            )
        )

    else:

        best_index = int(
            np.argmax(
                values
            )
        )


    best_epoch = (
        best_index
        +
        1
    )

    best_value = float(
        values[
            best_index
        ]
    )


    # --------------------------------------------------------
    # Save training history
    # --------------------------------------------------------

    pd.DataFrame(
        history.history
    ).to_csv(
        run_dir
        / "training_history.csv",

        index=False
    )


    # --------------------------------------------------------
    # Save INNER-VALIDATION predictions
    # No outer data involved.
    # --------------------------------------------------------

    if model_name == "interaction_dbnet":

        val_prob = np.asarray(
            model.predict(
                [
                    fold_data[
                        "X_val"
                    ],
                    fisher_val,
                ],
                batch_size=
                    GROUP_BATCH_SIZE,
                verbose=0
            ),
            dtype=np.float32
        )


        interaction_coefficients = (
            model
            .get_layer(
                "shared_rtgp_interaction_scorer"
            )
            .layer
            .get_weights()[0]
            .reshape(-1)
            .astype(np.float32)
        )


        np.savez_compressed(

            run_dir
            / "validation_predictions.npz",

            probabilities=
                val_prob,

            labels=
                fold_data[
                    "y_val"
                ].astype(
                    np.float32
                ),

            interaction_coefficients=
                interaction_coefficients,

            seed=
                np.asarray(
                    seed,
                    dtype=np.int64
                ),

            best_epoch=
                np.asarray(
                    best_epoch,
                    dtype=np.int64
                ),

            runtime_seconds=
                np.asarray(
                    runtime_seconds
                ),
        )


    elif model_name == "binary_dbnet":

        val_prob = np.asarray(
            model.predict(
                X_binary_val,
                batch_size=
                    BINARY_BATCH_SIZE,
                verbose=0
            ),
            dtype=np.float32
        )


        np.savez_compressed(

            run_dir
            / "validation_predictions.npz",

            probabilities=
                val_prob,

            labels=
                np.argmax(
                    y_binary_val_onehot,
                    axis=1
                ).astype(
                    np.int64
                ),

            seed=
                np.asarray(
                    seed,
                    dtype=np.int64
                ),

            best_epoch=
                np.asarray(
                    best_epoch,
                    dtype=np.int64
                ),

            runtime_seconds=
                np.asarray(
                    runtime_seconds
                ),
        )


    else:

        val_prob = np.asarray(
            model.predict(
                fold_data[
                    "X_val"
                ],
                batch_size=
                    GROUP_BATCH_SIZE,
                verbose=0
            ),
            dtype=np.float32
        )


        np.savez_compressed(

            run_dir
            / "validation_predictions.npz",

            probabilities=
                val_prob,

            labels=
                fold_data[
                    "y_val"
                ].astype(
                    np.float32
                ),

            seed=
                np.asarray(
                    seed,
                    dtype=np.int64
                ),

            best_epoch=
                np.asarray(
                    best_epoch,
                    dtype=np.int64
                ),

            runtime_seconds=
                np.asarray(
                    runtime_seconds
                ),
        )


    # --------------------------------------------------------
    # Completion metadata LAST
    #
    # This is deliberate:
    # training_summary.json means all required artifacts exist.
    # --------------------------------------------------------

    summary = {

        "status":
            "completed",

        "subject":
            subject,

        "outer_fold":
            int(
                fold
            ),

        "seed":
            int(
                seed
            ),

        "model":
            model_name,

        "parameter_count":
            int(
                model.count_params()
            ),

        "monitor":
            monitor,

        "monitor_mode":
            mode,

        "best_epoch":
            int(
                best_epoch
            ),

        "best_monitor_value":
            best_value,

        "runtime_seconds":
            float(
                runtime_seconds
            ),

        "batch_size":
            (
                BINARY_BATCH_SIZE
                if
                model_name
                ==
                "binary_dbnet"
                else
                GROUP_BATCH_SIZE
            ),

        "learning_rate":
            LEARNING_RATE,

        "maximum_epochs":
            MAX_EPOCHS,

        "patience":
            PATIENCE,

        "min_delta":
            MIN_DELTA,

        "training_characters_1based":
            fold_data[
                "train_chars"
            ],

        "inner_validation_characters_1based":
            fold_data[
                "val_chars"
            ],

        # Indices only, not data access
        "outer_characters_1based":
            fold_data[
                "outer_chars"
            ],

        "outer_test_used_for_training":
            False,

        "outer_test_used_for_model_selection":
            False,

        "outer_performance_evaluated":
            False,

        "completed_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


    atomic_json_save(
        run_dir
        / "training_summary.json",
        summary
    )


    save_progress(
        subject=subject,
        fold=fold,
        stage="neural_model_complete",
        seed=seed,
        model=model_name,
        status="completed"
    )


    if model_name == "binary_dbnet":

        print(
            f"  {model_name} seed {seed}: "
            f"{runtime_seconds/60:.2f} min, "
            f"best epoch {best_epoch}, "
            f"val AUC {best_value:.6f}"
        )

    else:

        print(
            f"  {model_name} seed {seed}: "
            f"{runtime_seconds/60:.2f} min, "
            f"best epoch {best_epoch}, "
            f"val loss {best_value:.6f}"
        )


    del model
    del history
    del val_prob

    gc.collect()
    tf.keras.backend.clear_session()


# ============================================================
# 16. CHECKPOINT LOAD AUDIT
# ============================================================

def audit_fold_checkpoints(
    subject,
    fold,
    beta
):

    print(
        "\nCHECKPOINT LOAD AUDIT"
    )


    for seed in SEEDS:

        for model_name in [
            "structured_dbnet",
            "glass_gated_dbnet",
            "interaction_dbnet",
            "binary_dbnet",
        ]:

            run_dir = (
                SUBJECT_RUN_ROOT
                / subject
                / f"outer_fold_{fold}"
                / f"seed_{seed}"
                / model_name
            )


            if not model_complete(
                run_dir,
                subject,
                fold,
                seed,
                model_name
            ):

                raise RuntimeError(
                    f"Incomplete model: "
                    f"{subject} fold {fold} "
                    f"seed {seed} {model_name}"
                )


            tf.keras.backend.clear_session()
            gc.collect()

            tf.keras.utils.set_random_seed(
                seed
            )


            model = construct_model(
                model_name,
                beta
            )


            assert (
                model.count_params()
                ==
                EXPECTED_PARAMS[
                    model_name
                ]
            )


            model.load_weights(
                run_dir
                / "best.weights.h5"
            )


            for variable in model.weights:

                if not np.isfinite(
                    variable.numpy()
                ).all():

                    raise RuntimeError(
                        "Non-finite checkpoint weight."
                    )


            print(
                f"PASS | seed {seed} | "
                f"{model_name}"
            )


            del model


    tf.keras.backend.clear_session()
    gc.collect()


# ============================================================
# 17. TRAIN ONE COMPLETE FOLD
# ============================================================

def run_one_fold(
    subject,
    fold,
    X_all,
    y_all,
):

    print("\n" + "=" * 100)
    print(
        f"{subject} OUTER FOLD {fold}"
    )
    print("=" * 100)


    time_guard(
        subject,
        fold,
        "fold_start"
    )


    fold_data = prepare_fold_data(
        X_all,
        y_all,
        fold
    )


    print(
        "Train:",
        fold_data[
            "train_chars"
        ]
    )

    print(
        "Inner val:",
        fold_data[
            "val_chars"
        ]
    )

    print(
        "Outer indices only:",
        fold_data[
            "outer_chars"
        ]
    )

    print(
        "Outer EEG/labels accessed: NO"
    )


    # --------------------------------------------------------
    # GLASS
    # --------------------------------------------------------

    beta = run_glass(
        subject,
        fold,
        fold_data
    )


    assert beta.shape == (
        8,
        128
    )


    # --------------------------------------------------------
    # Fisher features
    # --------------------------------------------------------

    print(
        "\nComputing training Fisher-z..."
    )

    fisher_train = (
        compute_fisher_features(
            fold_data[
                "X_train"
            ]
        )
    )


    print(
        "Computing validation Fisher-z..."
    )

    fisher_val = (
        compute_fisher_features(
            fold_data[
                "X_val"
            ]
        )
    )


    assert fisher_train.shape == (
        500,
        6,
        28
    )

    assert fisher_val.shape == (
        100,
        6,
        28
    )


    # --------------------------------------------------------
    # Binary data
    # --------------------------------------------------------

    X_binary_train = (
        fold_data[
            "X_train"
        ]
        .reshape(
            3000,
            8,
            128,
            1
        )
    )


    binary_train_labels = (
        fold_data[
            "y_train"
        ]
        .reshape(-1)
        .astype(
            np.int64
        )
    )


    y_binary_train_onehot = (
        tf.keras.utils
        .to_categorical(
            binary_train_labels,
            num_classes=2
        )
        .astype(
            np.float32
        )
    )


    X_binary_val = (
        fold_data[
            "X_val"
        ]
        .reshape(
            600,
            8,
            128,
            1
        )
    )


    binary_val_labels = (
        fold_data[
            "y_val"
        ]
        .reshape(-1)
        .astype(
            np.int64
        )
    )


    y_binary_val_onehot = (
        tf.keras.utils
        .to_categorical(
            binary_val_labels,
            num_classes=2
        )
        .astype(
            np.float32
        )
    )


    print(
        "\nBinary train:",
        X_binary_train.shape
    )

    print(
        "Targets/non-targets:",
        int(
            binary_train_labels.sum()
        ),
        int(
            len(
                binary_train_labels
            )
            -
            binary_train_labels.sum()
        )
    )


    # --------------------------------------------------------
    # Structured family
    # --------------------------------------------------------

    for seed in SEEDS:

        print("\n" + "-" * 80)
        print(
            f"STRUCTURED FAMILY — SEED {seed}"
        )
        print("-" * 80)


        for model_name in (
            STRUCTURED_ORDER[
                seed
            ]
        ):

            train_one_model(

                subject,
                fold,
                seed,
                model_name,
                beta,
                fold_data,
                fisher_train,
                fisher_val,
                X_binary_train,
                y_binary_train_onehot,
                X_binary_val,
                y_binary_val_onehot,
            )


    # --------------------------------------------------------
    # Binary family
    # --------------------------------------------------------

    for seed in SEEDS:

        print("\n" + "-" * 80)
        print(
            f"BINARY DBNET — SEED {seed}"
        )
        print("-" * 80)


        train_one_model(

            subject,
            fold,
            seed,
            "binary_dbnet",
            beta,
            fold_data,
            fisher_train,
            fisher_val,
            X_binary_train,
            y_binary_train_onehot,
            X_binary_val,
            y_binary_val_onehot,
        )


    # --------------------------------------------------------
    # Final checkpoint audit
    # --------------------------------------------------------

    audit_fold_checkpoints(
        subject,
        fold,
        beta
    )


    # --------------------------------------------------------
    # Mark fold complete
    # --------------------------------------------------------

    fold_manifest = {

        "status":
            "training_complete",

        "subject":
            subject,

        "outer_fold":
            int(
                fold
            ),

        "training_characters_1based":
            fold_data[
                "train_chars"
            ],

        "inner_validation_characters_1based":
            fold_data[
                "val_chars"
            ],

        "outer_characters_indices_only_1based":
            fold_data[
                "outer_chars"
            ],

        "glass_fit_complete":
            True,

        "neural_checkpoints_complete":
            12,

        "outer_test_eeg_accessed":
            False,

        "outer_performance_evaluated":
            False,

        "completed_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


    atomic_json_save(

        SUBJECT_RUN_ROOT
        / subject
        / f"outer_fold_{fold}"
        / "fold_training_complete.json",

        fold_manifest
    )


    # --------------------------------------------------------
    # Recovery ZIP immediately after fold
    # --------------------------------------------------------

    zip_path = snapshot_fold(
        subject,
        fold,
        suffix="complete"
    )


    save_progress(
        subject=subject,
        fold=fold,
        stage="fold_training_complete",
        status="completed"
    )


    print("\n" + "=" * 100)
    print(
        f"{subject} FOLD {fold} — TRAINING COMPLETE"
    )
    print("=" * 100)

    print(
        "Fold-local GLASS: PASS"
    )

    print(
        "All 12 neural checkpoints: PASS"
    )

    print(
        "Outer EEG/labels accessed: NO"
    )

    print(
        "Outer performance evaluated: NO"
    )

    print(
        "Recovery ZIP:",
        zip_path
    )


    # Free fold arrays
    del fold_data
    del fisher_train
    del fisher_val
    del X_binary_train
    del X_binary_val
    del y_binary_train_onehot
    del y_binary_val_onehot

    gc.collect()
    tf.keras.backend.clear_session()


# ============================================================
# 18. MAIN UNATTENDED QUEUE
# ============================================================

print("\n" + "=" * 100)
print("STARTING UNATTENDED TRAINING QUEUE")
print("=" * 100)

print(
    "Subjects:",
    SUBJECT_QUEUE
)

print(
    "Safe session limit:",
    SAFE_LIMIT_HOURS,
    "hours"
)

print(
    "Elapsed before training:",
    round(
        elapsed_hours(),
        2
    ),
    "hours"
)

print(
    "Outer performance evaluation: DISABLED"
)


save_progress(
    stage="unattended_queue_started",
    status="running"
)


for subject in SUBJECT_QUEUE:

    print("\n" + "#" * 100)
    print(
        f"LOADING {subject}"
    )
    print("#" * 100)


    X_subject, y_subject = (
        load_subject(
            subject
        )
    )


    print(
        "Prepared data:",
        X_subject.shape,
        y_subject.shape
    )


    for fold in range(
        1,
        8
    ):

        time_guard(
            subject,
            fold,
            "before_fold"
        )


        complete_marker = (
            SUBJECT_RUN_ROOT
            / subject
            / f"outer_fold_{fold}"
            / "fold_training_complete.json"
        )


        if complete_marker.is_file():

            print(
                f"\n{subject} Fold {fold}: "
                "ALREADY COMPLETE — SKIPPING"
            )

            continue


        run_one_fold(
            subject,
            fold,
            X_subject,
            y_subject,
        )


    del X_subject
    del y_subject

    gc.collect()


# ============================================================
# 19. FINAL STATUS
# ============================================================

save_progress(
    stage="queue_complete",
    status="completed"
)


print("\n" + "=" * 100)
print("UNATTENDED TRAINING QUEUE COMPLETE")
print("=" * 100)

print(
    "Elapsed:",
    round(
        elapsed_hours(),
        2
    ),
    "hours"
)

print(
    "Outer-test EEG accessed: NO"
)

print(
    "Outer performance evaluated: NO"
)

print("\nRecovery files:")
for p in sorted(
    RECOVERY_ROOT.glob("*.zip")
):
    print(" ", p)
